## 2023-BS-AI-017
Humna Imran

# Step 1: Import Required Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Step 2: Data Preprocessing

In [2]:
transform = transforms.Compose([
transforms.Grayscale(),
transforms.ToTensor()
])

# Step 3: Load Dataset

In [3]:
train_data = datasets.ImageFolder("/content/drive/My Drive/train", transform=transform)
test_data = datasets.ImageFolder("/content/drive/My Drive/test", transform=transform)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

# Step 4: Define CNN Model

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # First convolutional block
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Second convolutional block
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Adaptive pooling to ensure a fixed output size regardless of input image dimensions
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1)) # Output: (batch_size, 16, 1, 1)

        # Final fully connected layer, input size matches the output of adaptive_pool (16 features)
        self.fc = nn.Linear(16, 2) # Corrected: Output classes set to 2, matching existing data

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1) # Flatten to (batch_size, 16)
        x = self.fc(x)
        return x

# Step 5: Training the Model

In [5]:
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
for epoch in range(20):
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7417
Epoch 2, Loss: 0.6680
Epoch 3, Loss: 0.6797
Epoch 4, Loss: 0.6915
Epoch 5, Loss: 0.6461
Epoch 6, Loss: 0.6717
Epoch 7, Loss: 0.5871
Epoch 8, Loss: 0.6114
Epoch 9, Loss: 0.7998
Epoch 10, Loss: 0.7036
Epoch 11, Loss: 0.6922
Epoch 12, Loss: 0.6664
Epoch 13, Loss: 0.6732
Epoch 14, Loss: 0.6481
Epoch 15, Loss: 0.6110
Epoch 16, Loss: 0.6383
Epoch 17, Loss: 0.6325
Epoch 18, Loss: 0.6261
Epoch 19, Loss: 0.6609
Epoch 20, Loss: 0.6137


# Step 6: Model Evaluation

In [6]:
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print("Accuracy:", 100 * correct / total, "%")

Accuracy: 100.0 %


### Step 7: Test with a Custom Image

In [7]:
from PIL import Image
image_path = '/content/square.png'

try:
    img = Image.open(image_path).convert('L')
    img_tensor = transform(img).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        _, predicted_class = torch.max(output, 1)
    class_names = train_data.classes
    predicted_label = class_names[predicted_class.item()]

    print(f"The image is predicted to be: {predicted_label}")

except FileNotFoundError:
    print(f"Error: The file '{image_path}' was not found. Please ensure the image is uploaded and the path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")


The image is predicted to be: square


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
